## Как собрать оригинальный FA1?

Так как теперь поддерживается только улучшенный современный FA2, доставать эту бебру будем через Docker. Он позволяет создать изолированное окружение со старыми `torch` и `CUDA`, при этом все это сможет корректно работать на современных CUDA 13.2. 

### Что нужно для запуска этого кода (README):

1. Установленный для Windows **Docker** с возможностью *WSL Integration*

2. В терминале WSL скачайте образ:
```bash
docker run --rm --gpus all nvidia/cuda:11.6.2-base-ubuntu20.04 nvidia-smi
```

3. В папке FlashAttention-2 создай или используй `Dockerfile` в новой директории
```bash
mkdir fa1_benchmark && cd fa1_benchmark
nano Dockerfile
```

4. Там должно быть следующее:
```Dockerfile
# Берем официальный образ PyTorch времен выхода FlashAttention 1
FROM pytorch/pytorch:1.13.1-cuda11.6-cudnn8-devel

# Ставим утилиты для компиляции C++/CUDA
RUN apt-get update && apt-get install -y git build-essential
RUN pip install --upgrade pip
RUN pip install ninja packaging pandas jupyter

# Компилируем ОРИГИНАЛЬНЫЙ FlashAttention v1.0.9
# Флаг --no-build-isolation помогает избежать багов со старыми версиями pip
RUN pip install flash-attn==1.0.9 --no-build-isolation

# Настраиваем рабочую директорию
WORKDIR /workspace
```

5. В терминале (в той же папке fa1_benchmark) запусти сборку (рекомендую отключить VPN):
```bash
docker build -t flash_attn_v1_env .
```
**Внимание:** Сборка займет 10-15 минут, и потребует 10GB памяти. На этом этапе у меня закончилось терпение, но команды рабочие. Дело в том что установка flash-attn — это не скачивание питоновских скриптов, а жесткая компиляция .cu файлов.

6. Когда соберется, запускаем контейнер с пробросом GPU и портов для Jupyter:
```bash
docker run --gpus all -p 8888:8888 -v $(pwd):/workspace -it flash_attn_v1_env jupyter notebook --ip 0.0.0.0 --allow-root --no-browser
```

In [ ]:
import torch
import math
import pandas as pd
# Импортируем тот самый оригинальный FA1
from flash_attn.flash_attn_interface import flash_attn_qkvpacked_func

# 1. Настройки (Осторожно с N, стандартный метод съест всю VRAM)
B = 4          # Batch size
H = 12         # Number of heads
N = 4096       # Sequence length
d = 64         # Head dimension

# Оригинальный FA1 ожидает Q, K, V в одном упакованном тензоре
# формата: [Batch, SeqLen, 3, Heads, HeadDim]
qkv = torch.randn(B, N, 3, H, d, dtype=torch.float16, device='cuda')

# Распаковываем для классического Attention и транспонируем
# в классический формат: [Batch, Heads, SeqLen, HeadDim]
q = qkv[:, :, 0, :, :].transpose(1, 2)
k = qkv[:, :, 1, :, :].transpose(1, 2)
v = qkv[:, :, 2, :, :].transpose(1, 2)

# ==========================================
# ФУНКЦИИ ВЫЧИСЛЕНИЯ
# ==========================================
def standard_attention(q, k, v):
    # Классическая материализация матрицы N x N в HBM
    scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(d)
    p = torch.softmax(scores, dim=-1)
    out = torch.matmul(p, v)
    return out

def flash_attention_1(qkv):
    # Тот самый кастомный CUDA-кернел от Tri Dao
    out = flash_attn_qkvpacked_func(qkv, dropout_p=0.0, return_attn_probs=False)
    return out

# ==========================================
# БЕНЧМАРК (с использованием CUDA Events)
# ==========================================
def benchmark(func, args, num_runs=50):
    # Прогрев GPU
    for _ in range(10):
        _ = func(*args)
    torch.cuda.synchronize()

    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)
    
    start.record()
    for _ in range(num_runs):
        _ = func(*args)
    end.record()
    torch.cuda.synchronize()
    
    return start.elapsed_time(end) / num_runs

time_std = benchmark(standard_attention, (q, k, v))
time_fa1 = benchmark(flash_attention_1, (qkv,))

# ==========================================
# МЕТРИКИ И ВЫВОД
# ==========================================
bytes_per_elem = 2
hbm_std = (B * H * (4 * N * d + 4 * (N ** 2)) * bytes_per_elem) / (1024**3)
hbm_fa1 = (B * H * (4 * N * d) * bytes_per_elem) / (1024**3)

flops_matmul = 4 * B * H * (N ** 2) * d
gflops_std = (flops_matmul + 3 * B * H * (N ** 2)) / 10**9
gflops_fa1 = (flops_matmul + 5 * B * H * (N ** 2)) / 10**9

results = pd.DataFrame({
    'Metric': ['GFLOPs', 'HBM R/W (GB)', 'Runtime (ms)'],
    'Standard (PyTorch 1.13)': [f"{gflops_std:.1f}", f"{hbm_std:.3f}", f"{time_std:.2f}"],
    'Original FlashAttention-1': [f"{gflops_fa1:.1f}", f"{hbm_fa1:.3f}", f"{time_fa1:.2f}"]
})

print(f"Конфигурация: B={B}, H={H}, N={N}, d={d} | Hardware: RTX 3050 Ti (Ampere)\n")
print(results.to_string(index=False))